# grid2benchmark Tutorial

This notebook walks you through the full workflow of the **grid2benchmark** package — a standalone benchmarking harness for [Grid2Op](https://grid2op.readthedocs.io/) power-system reinforcement-learning algorithms.

**What you'll learn:**
1. Installation and setup
2. Importing the package
3. Core features and basic usage (Python API + CLI)
4. Configuration options (`BenchmarkConfig`, `ScenarioConfig`)
5. Handling input and output (topology / time-series files, result structure)
6. Error handling and debugging
7. Advanced usage — multiple scenarios, backend selection, custom agents

## 1. Installation and Setup

Install `grid2benchmark` and its dependencies with `uv` (recommended) or `pip`:

```bash
# Recommended: uv
uv add grid2benchmark

# Or pip
pip install grid2benchmark
```

> **Note:** `grid2op`, `pandapower`, `lightsim2grid`, and `pypowsybl2grid` are required dependencies installed automatically.

Verify the installation:

In [ ]:
import importlib.metadata
print("grid2benchmark version:", importlib.metadata.version("grid2benchmark"))

import grid2op
print("grid2op       version:", grid2op.__version__)

import pandapower as pp
print("pandapower    version:", pp.__version__)

## 2. Importing the Package

The main public API is available from the top-level `grid2benchmark` module:

In [ ]:
from pathlib import Path

from grid2benchmark import (
    run_benchmark,       # high-level orchestration function
    BenchmarkConfig,     # top-level config (max steps, KPIs, scenarios)
    ScenarioConfig,      # per-scenario config (env, topology, time-series, backend)
    TopologySource,      # points to an external grid topology file
    TimeSeriesSource,    # points to an external chronics directory
)

print("Imports OK")

## 3. Core Features and Basic Usage

### 3.1 Minimal benchmark run (built-in Grid2Op environment)

The simplest call uses Grid2Op's bundled `l2rpn_case14_sandbox` environment,
runs the first time-series episode for 10 steps, and evaluates three KPIs.

The `algorithm` argument can be:
- a `Path` to a `.py` file containing a `build_agent(env, context)` function, or
- a Python source-code string with the same function.

In [ ]:
ALGORITHM_SOURCE = """
class DoNothingAgent:
    def __init__(self, action_space):
        self._as = action_space

    def act(self, observation):
        return self._as()          # no-op action

def build_agent(env, context):
    return DoNothingAgent(env.action_space)
"""

result = run_benchmark(
    ALGORITHM_SOURCE,
    BenchmarkConfig(
        scenarios=[
            ScenarioConfig(
                env_name="l2rpn_case14_sandbox",
                time_series_ids=(0,),   # run only the first episode
            )
        ],
        max_steps=10,
        kpis=("survival", "violations", "latency"),
    ),
)

print("Top-level keys:", list(result.keys()))
print("Scenario count:", result["summary"]["scenario_count"])
print("Episode count :", result["summary"]["episode_count"])

### 3.2 Running from a file (the template algorithm)

The `examples/` directory ships two ready-to-use algorithms.
Pass a `Path` object to load an algorithm from disk:

In [ ]:
result_file = run_benchmark(
    Path("examples/algorithm_template.py"),
    BenchmarkConfig(
        scenarios=[ScenarioConfig(time_series_ids=(0,))],
        max_steps=5,
        kpis=("survival", "latency"),
    ),
)

# Episode detail from the first scenario
ep = result_file["scenarios"][0]["episodes"][0]
print(f"Steps completed : {ep['steps']}")
print(f"Runtime (s)     : {ep['runtime_seconds']:.4f}")
print(f"Terminated      : {ep['terminated']}")

### 3.3 CLI quick-start

You can drive benchmarks from a terminal without writing any Python:

```bash
grid2benchmark run \
  --algorithm examples/algorithm_template.py \
  --env l2rpn_case14_sandbox \
  --time-series 0 \
  --max-steps 10 \
  --kpis survival,violations,latency \
  --output results.json
```

The `results.json` file has the same structure as the Python API result.

## 4. Working with Configuration Options

### 4.1 `BenchmarkConfig`

| Field | Type | Default | Description |
|---|---|---|---|
| `scenarios` | `tuple[ScenarioConfig, ...]` | one default scenario | Scenario list |
| `max_steps` | `int` | `200` | Max steps per episode |
| `kpis` | `tuple[str, ...]` | all available | KPIs to compute |

Available KPI names:

In [ ]:
from grid2benchmark._config import AVAILABLE_KPI_NAMES, DEFAULT_MAX_STEPS

print("Available KPIs:")
for name in AVAILABLE_KPI_NAMES:
    print(" •", name)

print(f"\nDefault max steps: {DEFAULT_MAX_STEPS}")

### 4.2 `ScenarioConfig`

| Field | Type | Default | Description |
|---|---|---|---|
| `env_name` | `str` | `l2rpn_case14_sandbox` | Grid2Op environment name |
| `time_series_ids` | `tuple[int, ...] \| None` | `None` (all) | Which episodes to run |
| `topology` | `TopologySource \| None` | `None` | External topology file |
| `time_series` | `TimeSeriesSource \| None` | `None` | External chronics dir |
| `backend` | `str \| None` | `None` | Power-flow solver: `pandapower`, `lightsim2grid`, or `pypowsybl` |

In [ ]:
from grid2benchmark._config import SUPPORTED_BACKENDS

print("Supported backends:", SUPPORTED_BACKENDS)

# Selecting a subset of time-series episodes
sc = ScenarioConfig(
    env_name="l2rpn_case14_sandbox",
    time_series_ids=(0, 2, 4),   # only episodes 0, 2, and 4
)
print(f"\nScenario env  : {sc.env_name}")
print(f"Episode IDs   : {sc.time_series_ids}")
print(f"Backend       : {sc.backend!r}  (None = auto)")

### 4.3 Backend selection

When `backend` is `None` (the default):
- If a `topology` file is provided → `pandapower` is used automatically.
- If no topology is provided → Grid2Op's built-in default applies.

```python
ScenarioConfig(backend="pandapower")    # always use PandaPower
ScenarioConfig(backend="lightsim2grid") # fast C++ solver (requires lightsim2grid)
ScenarioConfig(backend="pypowsybl")     # PowerSyBl solver (requires pypowsybl2grid)
```

> If a backend package is not installed, a clear `ImportError` with an install command
> is raised **at runtime** — never at import time.

## 5. Handling Input and Output

### 5.1 External topology and time-series files

Provide your own pandapower grid JSON and a Grid2Op chronics directory instead of relying on a built-in environment:

In [ ]:
TOPOLOGY_FILE = Path("test_data/rte_case118_example/grid.json")
CHRONICS_DIR  = Path("test_data/rte_case118_example/chronics")

if TOPOLOGY_FILE.exists() and CHRONICS_DIR.exists():
    sc_external = ScenarioConfig(
        env_name="rte_case118_example",
        topology=TopologySource(
            format="pandapower",
            path=TOPOLOGY_FILE,
        ),
        time_series=TimeSeriesSource(
            format="grid2op_chronics_dir",
            path=CHRONICS_DIR,
        ),
        time_series_ids=(0,),
    )
    print("ScenarioConfig with external files:")
    print(f"  topology : {sc_external.topology.path}")
    print(f"  chronics : {sc_external.time_series.path}")
else:
    print("test_data not found — skipping.")
    print("Populate it by copying the bundled dataset:")
    print("  Copy-Item -Recurse .venv/Lib/site-packages/grid2op/data/rte_case118_example test_data/")

### 5.2 Scenario JSON file format (for `--scenarios`)

```json
[
    {
        "env_name": "l2rpn_case14_sandbox",
        "time_series_ids": [0, 1],
        "backend": "pandapower"
    },
    {
        "env_name": "rte_case118_example",
        "topology": { "format": "pandapower", "path": "./data/grid.json" },
        "time_series": { "format": "grid2op_chronics_dir", "path": "./data/chronics" },
        "time_series_ids": [0],
        "backend": "lightsim2grid"
    }
]
```

```bash
grid2benchmark run --algorithm algo.py --scenarios scenarios.json --output results.json
```

### 5.3 Understanding the output structure

```
result
├── scenarios
│   └── [0]
│       ├── scenario_index
│       ├── environment           env_name, backend, topology, time_series
│       ├── executed_time_series_ids
│       ├── episodes
│       │   └── [0]               episode_index, time_series_id, steps,
│       │                         overload_violations, runtime_seconds, terminated
│       └── kpis                  aggregated metrics over all episodes
└── summary
    ├── scenario_count
    ├── episode_count
    └── kpis                      mean/min/max/count per KPI across all scenarios
```

In [ ]:
import json

# Inspect result structure from the earlier minimal run
print("=== environment ===")
env_block = result["scenarios"][0]["environment"]
print(json.dumps(env_block, indent=2))

print("\n=== episode[0] ===")
print(json.dumps(result["scenarios"][0]["episodes"][0], indent=2))

print("\n=== summary KPIs ===")
print(json.dumps(result["summary"]["kpis"], indent=2))

## 6. Error Handling and Debugging

### 6.1 Validation errors at construction time

All config objects are `frozen` dataclasses that validate input immediately at construction:

In [ ]:
import os, tempfile

# Empty env_name
try:
    ScenarioConfig(env_name="")
except ValueError as e:
    print(f"[env_name]  {e}")

# Unsupported backend
try:
    ScenarioConfig(backend="fancy_solver")
except ValueError as e:
    print(f"[backend]   {e}")

# Unsupported topology format
try:
    TopologySource(format="matpower", path=Path("."))
except ValueError as e:
    print(f"[topo fmt]  {e}")

# Topology file must be .json (not .csv, .txt, etc.)
with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as f:
    tmp = Path(f.name)
try:
    TopologySource(format="pandapower", path=tmp)
except ValueError as e:
    print(f"[topo ext]  {e}")
finally:
    os.unlink(tmp)

# Negative time-series ID
try:
    ScenarioConfig(time_series_ids=(-1,))
except ValueError as e:
    print(f"[ts ids]    {e}")

# max_steps must be positive
try:
    BenchmarkConfig(max_steps=0)
except ValueError as e:
    print(f"[max_steps] {e}")

### 6.2 Algorithm file must define `build_agent`

In [ ]:
from grid2benchmark._algorithm import validate_algorithm
import types

# Simulate a module missing build_agent
bad_module = types.ModuleType("bad_algo")
try:
    validate_algorithm(bad_module)
except (AttributeError, ValueError) as e:
    print(f"Missing build_agent: {e}")

# Algorithm must also return an object with a callable act()
missing_act_source = """
def build_agent(env, context):
    class BadAgent:
        pass
    return BadAgent()
"""
try:
    run_benchmark(
        missing_act_source,
        BenchmarkConfig(
            scenarios=[ScenarioConfig(time_series_ids=(0,))],
            max_steps=1,
        ),
    )
except ValueError as e:
    print(f"Missing act():  {e}")

### 6.3 Backend package not installed

If a requested backend is not installed, a clear error is raised at runtime with an install hint:

```python
sc = ScenarioConfig(backend="lightsim2grid")
run_benchmark(algo, BenchmarkConfig(scenarios=[sc]))
# → ImportError: lightsim2grid is required for the 'lightsim2grid' backend.
#               Install it with: pip install lightsim2grid
```

### 6.4 Enabling verbose logging

```python
import logging
logging.basicConfig(level=logging.INFO)
```

## 7. Advanced Usage and Examples

### 7.1 Comparing two algorithms side-by-side

In [ ]:
DO_NOTHING_SOURCE = """
def build_agent(env, context):
    class Agent:
        def __init__(self, as_): self._as = as_
        def act(self, obs): return self._as()
    return Agent(env.action_space)
"""

config = BenchmarkConfig(
    scenarios=[ScenarioConfig(time_series_ids=(0, 1))],
    max_steps=15,
    kpis=("survival", "violations", "latency"),
)

result_noop   = run_benchmark(DO_NOTHING_SOURCE, config)
result_greedy = run_benchmark(Path("examples/greedy_baseline.py"), config)


def show_summary(name, res):
    kpis = res["summary"]["kpis"]
    print(f"\n{'─'*46}")
    print(f"  {name}")
    print(f"{'─'*46}")
    for k, v in kpis.items():
        print(f"  {k:36s}  mean={v['mean']:.4f}")

show_summary("Do-Nothing agent", result_noop)
show_summary("Greedy baseline",  result_greedy)

### 7.2 Multiple scenarios in one benchmark run

Run the same algorithm against different episode subsets — results are aggregated together in `summary`:

In [ ]:
multi_config = BenchmarkConfig(
    scenarios=[
        ScenarioConfig(env_name="l2rpn_case14_sandbox", time_series_ids=(0,)),
        ScenarioConfig(env_name="l2rpn_case14_sandbox", time_series_ids=(1,)),
    ],
    max_steps=10,
    kpis=("survival", "latency"),
)

result_multi = run_benchmark(DO_NOTHING_SOURCE, multi_config)

print(f"Total scenarios : {result_multi['summary']['scenario_count']}")
print(f"Total episodes  : {result_multi['summary']['episode_count']}")
for sc_res in result_multi["scenarios"]:
    idx = sc_res["scenario_index"]
    ts  = sc_res["executed_time_series_ids"]
    ep  = sc_res["episodes"][0]
    print(f"  Scenario {idx}  ts={ts}  steps={ep['steps']}  runtime={ep['runtime_seconds']:.4f}s")

### 7.3 External grid file — rte_case118_example

The 118-bus RTE case ships with Grid2Op and can be used as a realistic benchmark:

In [ ]:
TOPOLOGY_FILE = Path("test_data/rte_case118_example/grid.json")
CHRONICS_DIR  = Path("test_data/rte_case118_example/chronics")

if TOPOLOGY_FILE.exists() and CHRONICS_DIR.exists():
    ext_config = BenchmarkConfig(
        scenarios=[
            ScenarioConfig(
                env_name="rte_case118_example",
                topology=TopologySource(format="pandapower", path=TOPOLOGY_FILE),
                time_series=TimeSeriesSource(format="grid2op_chronics_dir", path=CHRONICS_DIR),
                time_series_ids=(0,),
                backend="pandapower",
            )
        ],
        max_steps=20,
        kpis=("survival", "violations", "latency"),
    )

    result_ext = run_benchmark(Path("examples/algorithm_template.py"), ext_config)
    env_info   = result_ext["scenarios"][0]["environment"]

    print("env_name  :", env_info["env_name"])
    print("backend   :", env_info["backend"])
    print("topology  :", env_info["topology"]["path"])
    print("\nKPIs:")
    for kpi, vals in result_ext["summary"]["kpis"].items():
        print(f"  {kpi:36s}  mean={vals['mean']:.4f}")
else:
    print("test_data not available. Populate it with:")
    print("  Copy-Item -Recurse .venv/Lib/site-packages/grid2op/data/rte_case118_example test_data/")

### 7.4 Writing a custom agent

An algorithm file only needs one function: `build_agent(env, context)` returning any object with an `act(observation)` method.

```python
# my_agent.py

class LineReconnectAgent:
    """Reconnects the first disconnected line it finds; otherwise takes no action."""

    def __init__(self, env):
        self._action_space = env.action_space

    def act(self, observation):
        # Try to reconnect any disconnected line
        for i, status in enumerate(observation.line_status):
            if not status:
                return self._action_space({"set_line_status": [(i, 1)]})
        return self._action_space()   # no-op


def build_agent(env, context):
    # context contains benchmark meta-data:
    # context["benchmark"]["max_steps"]        int
    # context["benchmark"]["kpis"]             list[str]
    # context["scenario"]["env_name"]          str
    # context["scenario"]["backend"]           str | None
    # context["scenario"]["topology"]          dict | None
    # context["scenario"]["time_series"]       dict | None
    # context["scenario"]["time_series_ids"]   list[int]
    return LineReconnectAgent(env)
```

### 7.5 Backend performance comparison

Benchmark the same algorithm with different backends to measure solver overhead:

In [ ]:
import time

FAST_ALGO = """
def build_agent(env, context):
    class A:
        def __init__(self, s): self._s = s
        def act(self, o): return self._s()
    return A(env.action_space)
"""

# Extend this list with "lightsim2grid" or "pypowsybl" if those packages are installed.
backends_to_test = ["pandapower"]

for backend in backends_to_test:
    sc  = ScenarioConfig(env_name="l2rpn_case14_sandbox", time_series_ids=(0,), backend=backend)
    cfg = BenchmarkConfig(scenarios=[sc], max_steps=20, kpis=("latency",))
    t0  = time.perf_counter()
    try:
        r = run_benchmark(FAST_ALGO, cfg)
        elapsed      = time.perf_counter() - t0
        latency_mean = r["summary"]["kpis"].get("latency", {}).get("mean", "n/a")
        print(f"  {backend:15s}  wall={elapsed:.3f}s  latency_mean={latency_mean}")
    except ImportError as e:
        print(f"  {backend:15s}  skipped (not installed): {e}")

---

## Summary

| Task | API |
|---|---|
| Minimal benchmark run | `run_benchmark(algo_str, BenchmarkConfig(...))` |
| Load algorithm from file | `run_benchmark(Path("algo.py"), ...)` |
| Limit episodes | `ScenarioConfig(time_series_ids=(0, 1, 2))` |
| External topology | `TopologySource(format="pandapower", path=Path("grid.json"))` |
| External time-series | `TimeSeriesSource(format="grid2op_chronics_dir", path=Path("chronics/"))` |
| Choose backend | `ScenarioConfig(backend="pandapower" \| "lightsim2grid" \| "pypowsybl")` |
| Multiple scenarios | `BenchmarkConfig(scenarios=[sc1, sc2, ...])` |
| CLI | `grid2benchmark run --algorithm algo.py --env ... --backend ... --output out.json` |
| Scenario file (CLI) | `grid2benchmark run --algorithm algo.py --scenarios scenarios.json` |

**Next steps:**
- Implement your own agent in `my_agent.py` and pass it to `run_benchmark`
- Try `lightsim2grid` for significantly faster power-flow computation
- Explore the `examples/` directory for ready-made algorithm templates
- See `README.md` for the full API reference and output schema